### sklearn  鸢尾花（Iris）数据集

In [8]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 加载鸢尾花数据集
iris = load_iris()
X = iris.data  # 特征
y = iris.target  # 标签

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 标准化特征值
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 初始化逻辑回归模型，指定multi_class参数为'multinomial'
model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=200)

# 训练模型
model.fit(X_train, y_train)

# 使用训练好的模型进行预测
y_pred = model.predict(X_test)

# 计算准确率
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# 打印详细的分类报告
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Accuracy: 1.0000
Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



/home/xr/.conda/envs/d2l/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


### 神经网络

In [ ]:
import torch
from torch import nn, optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [13]:
# 加载鸢尾花数据集
iris = load_iris()
X = iris.data
y = iris.target

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 标准化特征值
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 转换为PyTorch张量
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)
print(X_train.shape)
print(y_test.bincount())

torch.Size([120, 4])
tensor([10,  9, 11])


In [3]:
# class Net(nn.Module):
#     def __init__(self):
#         super(Net, self).__init__()
#         self.fc1 = nn.Linear(4, 100)  # 输入层到隐藏层
#         self.fc2 = nn.Linear(100, 100)  # 隐藏层到隐藏层
#         self.fc3 = nn.Linear(100, 3)   # 隐藏层到输出层
    
#     def forward(self, x):
#         x = torch.relu(self.fc1(x))
#         x = torch.relu(self.fc2(x))
#         x = self.fc3(x)
#         return x
# 初始化网络
# net = Net()   
net= nn.Sequential(
    nn.Linear(4, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 3),
)
# net(torch.randn(1, 4))


tensor([[-0.0760,  0.1072,  0.0479]], grad_fn=<AddmmBackward0>)

In [4]:
criterion = nn.CrossEntropyLoss()  # 损失函数
optimizer = optim.Adam(net.parameters(), lr=0.01)  # 优化器

In [5]:
epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()  # 清除梯度
    out = net(X_train)  # 前向传播
    loss = criterion(out, y_train)  # 计算损失
    loss.backward()  # 反向传播
    optimizer.step()  # 更新权重
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')

Epoch 10/100, Loss: 0.1330602467060089
Epoch 20/100, Loss: 0.04574160650372505
Epoch 30/100, Loss: 0.04091476649045944
Epoch 40/100, Loss: 0.038083966821432114
Epoch 50/100, Loss: 0.03317068889737129
Epoch 60/100, Loss: 0.026365121826529503
Epoch 70/100, Loss: 0.018587762489914894
Epoch 80/100, Loss: 0.01005656085908413
Epoch 90/100, Loss: 0.00439626257866621
Epoch 100/100, Loss: 0.002042504260316491


In [6]:
with torch.no_grad():
    output = net(X_test)
    _, predicted = torch.max(output.data, 1)
    accuracy = (predicted == y_test).sum().item() / y_test.size(0)
    print(f"Accuracy: {accuracy * 100}%")

Accuracy: 96.66666666666667%
